# Summary Markets

Provider-agnostic ticker universe table. One row per instrument with a `yahoo_ticker` column
pre-translated for yfinance. Rebuilt via `uv run irp` → Steps → `markets`.

Translation rules applied at build time:
- Currencies 6-char alpha (`EURUSD`) → `EURUSD=X`
- Currencies non-standard (`NOK_I`, `EUR_I`) → NULL (Stooq-specific, no Yahoo equivalent)
- Stooq stocks indices (`^_UK`, `^_US`) → NULL (Stooq-proprietary basket indices)
- Preferred shares `BASE_X` → `BASE-PX` (e.g. `RNR_F` → `RNR-PF`)
- All others → same as `Ticker`

In [1]:
import pandas as pd
from IPython.display import display

from irp.data._common import db
from irp.data.markets import markets

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

## Table

Schema, row count, and key stats for the `markets` table.
Rebuilt on demand — does not update automatically when sources are stored.

### `markets`

One row per instrument (canonical key = Stooq ticker symbol).

| Column | Type | Notes |
|---|---|---|
| Ticker | VARCHAR | Canonical key — Stooq symbol, matches all other tables |
| Market | VARCHAR | Market category from Stooq zip structure |
| yahoo_ticker | VARCHAR | yfinance symbol; NULL when no Yahoo equivalent |

In [2]:
_sample = markets('AAPL')
display(_sample.dtypes.to_frame('dtype'))
display(_sample.T)

,dtype
Ticker,str
Market,str
yahoo_ticker,str


,0
Ticker,AAPL
Market,nasdaq stocks
yahoo_ticker,AAPL


In [3]:
_stats = db().execute("""
    SELECT
        COUNT(*)                        AS tickers,
        COUNT(DISTINCT Market)          AS markets,
        SUM(yahoo_ticker IS NOT NULL)   AS have_yahoo_ticker,
        SUM(yahoo_ticker IS NULL)       AS null_yahoo_ticker
    FROM markets
""").df().T
_stats.columns = ['markets']
display(_stats)

,markets
tickers,14532.0
markets,11.0
have_yahoo_ticker,14503.0
null_yahoo_ticker,29.0


## Coverage by Market

How many tickers per market have a valid `yahoo_ticker` vs NULL.

In [4]:
display(db().execute("""
    SELECT
        Market,
        COUNT(*)                                                    AS tickers,
        SUM(yahoo_ticker IS NOT NULL)                               AS have_yahoo,
        SUM(yahoo_ticker IS NULL)                                   AS no_yahoo,
        ROUND(SUM(yahoo_ticker IS NOT NULL) * 100.0 / COUNT(*), 1) AS pct_yahoo
    FROM markets
    GROUP BY Market
    ORDER BY tickers DESC
""").df())

,Market,tickers,have_yahoo,no_yahoo,pct_yahoo
0,nasdaq stocks,4643,4643.0,0.0,100.0
1,nyse stocks,3670,3670.0,0.0,100.0
2,nyse etfs,2552,2552.0,0.0,100.0
3,currencies,1822,1806.0,16.0,99.1
4,nasdaq etfs,938,938.0,0.0,100.0
5,nysemkt stocks,306,306.0,0.0,100.0
6,bonds,284,284.0,0.0,100.0
7,cryptocurrencies,215,215.0,0.0,100.0
8,indices,62,62.0,0.0,100.0
9,money market,27,27.0,0.0,100.0


## Translation Breakdown

How many tickers fall into each translation category.

In [5]:
display(db().execute("""
    SELECT
        CASE
            WHEN yahoo_ticker IS NULL              THEN 'null (no Yahoo equivalent)'
            WHEN yahoo_ticker LIKE '%=X'           THEN 'currency (=X suffix)'
            WHEN yahoo_ticker LIKE '%-P_'          THEN 'preferred share (-PX)'
            WHEN yahoo_ticker = Ticker             THEN 'unchanged'
            ELSE                                        'other'
        END                                            AS translation,
        COUNT(*)                                       AS tickers
    FROM markets
    GROUP BY 1
    ORDER BY tickers DESC
""").df())

,translation,tickers
0,unchanged,12352
1,currency (=X suffix),1806
2,preferred share (-PX),345
3,null (no Yahoo equivalent),29


## NULL yahoo_ticker Detail

Which instruments have no Yahoo equivalent and why.

In [6]:
display(db().execute("""
    SELECT
        Market,
        COUNT(*) AS null_tickers,
        STRING_AGG(Ticker, ', ' ORDER BY Ticker) AS examples
    FROM markets
    WHERE yahoo_ticker IS NULL
    GROUP BY Market
    ORDER BY null_tickers DESC
""").df())

,Market,null_tickers,examples
0,currencies,16,"AUD_I, CAD_I, CHF_I, CZK_I, EUR_I, GBP_I, HUF_..."
1,stooq stocks indices,13,"^_DE, ^_HK, ^_HU, ^_JP, ^_PL, ^_PL20, ^_PLNC, ..."


## Translated Tickers Sample

Sample of tickers where `yahoo_ticker` differs from `Ticker`.

In [7]:
print('=== Currency translations (first 10) ===')
display(db().execute("""
    SELECT Ticker, Market, yahoo_ticker
    FROM markets
    WHERE yahoo_ticker LIKE '%=X'
    ORDER BY Ticker
    LIMIT 10
""").df())

print('=== Preferred share translations (first 10) ===')
display(db().execute("""
    SELECT Ticker, Market, yahoo_ticker
    FROM markets
    WHERE yahoo_ticker LIKE '%-P_'
    ORDER BY Ticker
    LIMIT 10
""").df())

=== Currency translations (first 10) ===


,Ticker,Market,yahoo_ticker
0,ARSAUD,currencies,ARSAUD=X
1,ARSBRL,currencies,ARSBRL=X
2,ARSBTC,currencies,ARSBTC=X
3,ARSCAD,currencies,ARSCAD=X
4,ARSCHF,currencies,ARSCHF=X
5,ARSCLP,currencies,ARSCLP=X
6,ARSCNY,currencies,ARSCNY=X
7,ARSCZK,currencies,ARSCZK=X
8,ARSDKK,currencies,ARSDKK=X
9,ARSEGP,currencies,ARSEGP=X


=== Preferred share translations (first 10) ===


,Ticker,Market,yahoo_ticker
0,ABR_D,nyse stocks,ABR-PD
1,ABR_E,nyse stocks,ABR-PE
2,ABR_F,nyse stocks,ABR-PF
3,ACP_A,nyse stocks,ACP-PA
4,ACR_C,nyse stocks,ACR-PC
5,ACR_D,nyse stocks,ACR-PD
6,ADC_A,nyse stocks,ADC-PA
7,AGM_D,nyse stocks,AGM-PD
8,AGM_E,nyse stocks,AGM-PE
9,AGM_F,nyse stocks,AGM-PF


## Market Distribution

Ticker count per market category.

In [8]:
display(db().execute("""
    SELECT
        Market,
        COUNT(*)                                    AS tickers,
        ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 1) AS pct_of_total
    FROM markets
    GROUP BY Market
    ORDER BY tickers DESC
""").df())

,Market,tickers,pct_of_total
0,nasdaq stocks,4643,32.0
1,nyse stocks,3670,25.3
2,nyse etfs,2552,17.6
3,currencies,1822,12.5
4,nasdaq etfs,938,6.5
5,nysemkt stocks,306,2.1
6,bonds,284,2.0
7,cryptocurrencies,215,1.5
8,indices,62,0.4
9,money market,27,0.2
